# Prop-DeOccNet Training — Google Colab

**Sebelum mulai:** Pastikan GPU aktif → `Runtime > Change runtime type > T4 GPU`

---
## Step 1 — Cek GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 2 — Mount Google Drive

Dataset dan project harus ada di Google Drive.
Struktur yang diharapkan di Drive:
```
MyDrive/
└── labeling-daun-itoh/
    ├── images/
    ├── output/coco/train.json
    ├── output/coco/val.json
    ├── output/coco/test.json
    ├── training/
    └── requirements-train.txt
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — Upload Project ke Drive

**Cara upload (lakukan sekali dari PC lokal):**
1. Zip folder project: klik kanan `labeling-daun-itoh` → Compress → `labeling-daun-itoh.zip`
2. Buka [drive.google.com](https://drive.google.com)
3. Upload `labeling-daun-itoh.zip` ke `My Drive`
4. Jalankan cell di bawah untuk unzip

In [ ]:
import os

PROJECT_ZIP = '/content/drive/MyDrive/labeling-daun-itoh.zip'
PROJECT_DIR = '/content/labeling-daun-itoh'

if not os.path.exists(PROJECT_DIR):
    print('Extracting project...')
    !unzip -q "{PROJECT_ZIP}" -d /content/
    print('Done!')
else:
    print('Project already extracted.')

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 4 - Install Dependencies

> Setelah cell ini selesai: **Runtime > Restart session**, lalu lanjut ke Step 5.
> Jangan install ulang torch � Colab sudah punya versi yang cocok dengan numpy-nya.


In [ ]:
# JANGAN install ulang torch/torchvision!
# PyTorch wheel cu121 dikompilasi SEBELUM numpy 2.0 release, sehingga tidak kompatibel
# dengan numpy 2.x yang ada di Colab -> _ARRAY_API not found -> torch.from_numpy crash.
# Gunakan torch bawaan Colab yang sudah sesuai dengan numpy-nya.

import torch
print(f"Torch (pre-installed): {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")

# Cython 3+ wajib untuk build pycocotools yang support numpy 2.x
!pip install "cython>=3.0.0" -q

# Build pycocotools dari source terhadap numpy saat ini (bukan binary numpy 1.x)
!pip install pycocotools --no-binary pycocotools --no-cache-dir -q

# Paket lainnya
!pip install albumentations==1.4.3 "scipy>=1.14.0" pyyaml tqdm -q

# Verifikasi
import numpy as np
import pycocotools._mask  # akan error di sini kalau masih ada masalah
print(f"numpy       : {np.__version__}")
print("pycocotools : OK")
print()
print("=== Done! Sekarang: Runtime > Restart session, lalu lanjut ke Step 5. ===")

## Step 5 — Verifikasi Dataset

In [ ]:
import json, os

for split in ['train', 'val', 'test']:
    path = f'output/coco/{split}.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'{split}: {len(data["images"])} images, {len(data["annotations"])} annotations')
    else:
        print(f'MISSING: {path}')
        print('  → Ekspor dulu dari Phase 1 tool: Menu Export > COCO JSON')

n_images = len([f for f in os.listdir('images') if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'\nTotal images di folder images/: {n_images}')

## Step 6 — Muat Checkpoints dari Drive (Wajib)

**Jalankan selalu sebelum training.** Checkpoint di-copy otomatis dari Drive ke Colab.
Kalau belum ada checkpoint, training dimulai dari awal secara otomatis.

> **Catatan:** Selama training, setiap checkpoint langsung di-sync ke Drive secara otomatis.
> Jika runtime putus, cukup jalankan ulang notebook dari Step 1 s/d Step 7 — training akan lanjut dari epoch terakhir.

In [ ]:
import os, shutil
from pathlib import Path

DRIVE_CKPT = Path('/content/drive/MyDrive/labeling-daun-itoh/checkpoints')
LOCAL_CKPT = Path('checkpoints')
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)

if DRIVE_CKPT.exists():
    pth_files = sorted(DRIVE_CKPT.glob('*.pth'))
    if pth_files:
        for f in pth_files:
            dst = LOCAL_CKPT / f.name
            shutil.copy2(f, dst)
        print(f'Loaded {len(pth_files)} checkpoint(s) from Drive:')
        for f in sorted(LOCAL_CKPT.glob('*.pth')):
            size_mb = f.stat().st_size / 1e6
            print(f'  {f.name} ({size_mb:.0f} MB)')
    else:
        print('Drive/checkpoints exists but is empty — training will start fresh')
else:
    print('No Drive checkpoints found — training will start fresh')


## Step 7 — Training

Menggunakan  (full config: ResNet-101, 512px, 50 epoch).
Estimasi waktu di T4 GPU: **4–8 jam** untuk 50 epoch.

**Auto-resume:** Training otomatis melanjutkan dari checkpoint terakhir yang ada di folder .
**Auto-sync ke Drive:** Setiap checkpoint langsung disalin ke Drive setelah disimpan, tanpa perlu menunggu Step 10.


In [ ]:
import sys
sys.path.insert(0, '/content/labeling-daun-itoh')

from training.train import train
train(
    'training/config_train.yaml',
    drive_checkpoint_dir='/content/drive/MyDrive/labeling-daun-itoh/checkpoints',
)


## Step 8 — TensorBoard (monitoring training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## Step 9 — Evaluasi Model Terbaik

In [ ]:
import torch, yaml
from torch.utils.data import DataLoader
from training.model import PropDeOccNet
from training.dataset import DaunDataset, collate_fn, get_val_transforms
from training.evaluate import evaluate

cfg = yaml.safe_load(open('training/config_train.yaml'))

model = PropDeOccNet(
    num_classes=cfg['num_classes'],
    backbone=cfg['backbone'],
    aspp_rates=cfg['aspp_rates'],
    use_boundary_head=cfg['use_boundary_head'],
)

ckpt = torch.load('checkpoints/best.pth', map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model = model.to('cuda')

test_ds = DaunDataset(cfg['test_json'], cfg['images_dir'], get_val_transforms(cfg['image_size']))
test_loader = DataLoader(test_ds, batch_size=1, collate_fn=collate_fn)

metrics = evaluate(model, test_loader, device='cuda')
print('\n=== Test Set Results ===')
for k, v in metrics.items():
    print(f'  {k:15s}: {v:.4f}')

## Step 10 — Simpan Hasil ke Google Drive

**Penting:** Colab akan reset setelah sesi berakhir. Simpan checkpoint ke Drive!

In [ ]:
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/labeling-daun-itoh'
os.makedirs(DRIVE_SAVE, exist_ok=True)

# Simpan checkpoints
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', f'{DRIVE_SAVE}/checkpoints', dirs_exist_ok=True)
    print('Checkpoints saved to Drive')

# Simpan TensorBoard runs
if os.path.exists('runs'):
    shutil.copytree('runs', f'{DRIVE_SAVE}/runs', dirs_exist_ok=True)
    print('TensorBoard runs saved to Drive')

print('Done! Files tersimpan di Google Drive.')

## Step 11 — (Opsional) Studi Ablasi M0–M3

In [ ]:
from training.ablation import run_ablation
run_ablation('training/config_train.yaml')

# Simpan hasil ablasi ke Drive
import shutil
shutil.copy('checkpoints/ablation_results.json',
            '/content/drive/MyDrive/labeling-daun-itoh/ablation_results.json')
print('Ablation results saved to Drive')